## Print information about a tuned chorale
1. Chorale name, tolerance, tonal diamond shape, limit max
1. Cent values, note names, scores, and ratios for every chord
2. Top notes cents, note names, cent values


In [1]:
import os
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [2]:
import logging, os, sys, time
from importlib import reload
import numpy as np
from importlib import reload
from collections import Counter, defaultdict
user = 'prent'
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')
base_dir = local_dir
WAVE_DIR = os.path.join(base_dir, 'Music', 'sflib')
# The latest files are here: Archive/straw-man/t1_r1.75_s2.50_md28_sn10/bwv253-opt.npy
numpy_dir = os.path.join(base_dir, 'Archive', 'straw-man')

np.set_printoptions(legacy='1.25')
import diamond_music_utils as dmu
import adaptive_tuning_util as atu 
from itertools import count, combinations, permutations
dmu.start_logger('test.log',log_level = 'info') # how to modify this so that it only prints to the log and not in the notebook.
logging.info(f'{base_dir = }, {numpy_dir = }, {WAVE_DIR = }')
rng = np.random.default_rng()

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [3]:
def print_chords(version, input_file, numpy_dir, measure, tolerance, ratios=True, print_individual_chords=True,\
            offset=0, use_werck_top_notes=False, print_top_notes = True):
    
    _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)
    try:
        floating_cents = np.load(input_file)
        existing_chorale_in_cents = np.rint(floating_cents).astype(int)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    if use_werck_top_notes:
        input_file = os.path.join(numpy_dir, f'{version}-w-top_notes.npy')
    else: 
        input_file = os.path.join(numpy_dir, f'{version}top-notes.npy')
        if print_top_notes:
            print(f'Loaded top_notes from {input_file = }')
    try:      
        top_notes = np.load(input_file)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    top_notes[1] = top_notes[1] + offset
    
    if print_top_notes:
        print(f'Key: {keys[root]} {mode}, {tolerance = }')
        print(f'\ntop notes:')
        print(*[inx for inx in np.arange(12)], sep='\t')
        print(*[note for note in top_notes[0]], sep = '\t')
        print(*[keys[note] for note in top_notes[0]], sep = '\t')
        print(*[cent_value for cent_value in top_notes[1]], sep = '\t')
    if print_individual_chords: 
        print(f'\n#          cents       note names   chord score')
        # #     +----- cents -----+--- note names---+--- chord score'
        # 0:    0  386    0  884	C♮ E♮ C♮ A♮	47.0
    if measure > 0: print(f'\nprinting only measure {measure}')
    prev_chord = np.zeros(4, dtype=int)
    header1 = f" # Fr/To Cents Ratio\t # Fr/To Cents Ratio\t # Fr/To Cents Ratio"
    
    for inx, chord_in_cents, chord_12 in zip(count(0,1), existing_chorale_in_cents.T, chorale.T):
        if not np.array_equal(prev_chord, chord_in_cents):
            if measure == 0 or 16 * (measure - 1) <= inx < 16 * measure:
                if print_individual_chords: 
                        # Join the note names into a single space-separated string to avoid numpy array formatting
                        pitches = ' '.join(map(str, keys[chord_12 % 12]))
                        print(f'{inx}: {atu.format_chord(chord_in_cents,4)}\t{pitches}\t{chord_scorer.score_chord(chord_in_cents, tolerance=tolerance)}')
                if ratios:
                    print(f'{header1}')
                    intervals = []
                    for inx1, inx2 in combinations(np.arange(4),2):
                            cent_value_interval_pair = np.array([chord_in_cents[inx1], chord_in_cents[inx2]])
                            cent_value_delta, cent_value_moves, cent_value_target = atu.cent_value_interval(cent_value_interval_pair)
                            best_idx = chord_scorer.find_best_interval(cent_value_delta, tolerance)[0]
                            ratio = str(atu.limit_format(tonal_diamond[best_idx])[0]).strip()
                            n1 = keys[chord_12[inx1] % 12]
                            n2 = keys[chord_12[inx2] % 12]
                            intervals.append((n1, n2, cent_value_delta, ratio))

                    def fmt(iv, idx):
                            n1, n2, cents, ratio = iv
                            return f"{idx:>2} {n1:>2} {n2:>2} {cents:>5} {ratio:^6}"

                    # print first and last three intervals on separate lines, nicely aligned and without Python punctuation
                    
                    print("   ".join(fmt(iv, i+1) for i, iv in enumerate(intervals[:3])))
                    print("   ".join(fmt(iv, i+1+3) for i, iv in enumerate(intervals[3:])))
        prev_chord = chord_in_cents.copy()
    return keys, root, mode

In [4]:
print(f'{numpy_dir = }')

numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man'


In [10]:
# python Straw_man_tuning_v3.py --chorale_list bwv253 bwv254 bwv255 bwv256 bwv257 bwv258 bwv259 bwv260 bwv261 bwv262 bwv263 bwv264 --limit_max 17 --tolerance 1 --ratio_factor 1.25 --max_delta 33 --rolls 5 --sa_iters 40 --sa_max_alpha 8.0 --sa_restarts 12 --parallel_restarts 6 --restart_repeat_threshold 2 --no-print_values --numpy_dir Archive/straw-man
# output: Archive/straw-man/bwv2??-opt.npy
ratio_factors = np.array([ "1.25"]) # , "1.25", "1.75"
stability_factors = np.array(["0"]) # , "1.25"
max_delta = 33
snaps = np.array([0])
suffixes = np.array(['-opt.npy']) # '-opt.npy',
# suffixes = np.array(['-sa-opt.npy', '-trans-sa-opt.npy'])
limit_max = 17
tolerance = 1
measure = 0 # 0 means print all measures
print_individual_chords = False
use_werck_top_notes = False
ratios = False
print_top_notes = False
print_hits_misses = False
total_scores = 0
num_scores = 0
max_score = 0
tonal_diamond = atu.build_tonal_diamond(limit_max)
chord_scorer = atu.ChordScorer(tonal_diamond)

chord_scorer.reset_cache()
for tolerance in [1]:
    for ratio_factor in ratio_factors:
        for stability_factor in stability_factors:
            for snap in snaps:
                for suffix in suffixes:
                    local_numpy_dir = numpy_dir #
                    print(f'{local_numpy_dir = }') 
                    print(f'{tolerance = }, {ratio_factor = }, {stability_factor = }, {snap = }, {suffix = }')
                    for version in ['bwv253', 'bwv254', 'bwv255', 'bwv256', 'bwv257', 'bwv258', 'bwv259', 'bwv260',  'bwv261', 'bwv262', 'bwv263', 'bwv264']: # ['bwv256']: #
                        try:
                            input_file = os.path.join(local_numpy_dir, f'{version}{suffix}') 
                            # print(f'{input_file = }')
                            existing_chorale_in_cents = np.load(input_file)
                            logging.info(f'{input_file = }')
                        except:
                            print(f'Trouble loading {input_file = }')
                            continue
                        num_scores += 1
                        scores = np.array([chord_scorer.score_chord(chord, tolerance=tolerance) for chord in existing_chorale_in_cents.T])
                        print(f'\nversion: {version}, Tol: {tolerance}, RF: {ratio_factor}, Average score: {round(np.average(scores),1)}, max score: {np.max(scores)} max chord: {np.argmax(scores)}')
                        total_scores += np.average(scores)
                        max_score = np.max([max_score, np.max(scores) ])
                        keys, root, mode = print_chords(version, input_file, local_numpy_dir, measure, tolerance, \
                                ratios=ratios, print_individual_chords=print_individual_chords, \
                                use_werck_top_notes=use_werck_top_notes, print_top_notes = print_top_notes)
    if print_hits_misses:
        print(f'hits and misses: {chord_scorer.return_cache_results()}')
    print(f'overall total: {round(total_scores,1)}, {num_scores = }, Average Score: {round(np.average(total_scores/num_scores),1)}, {max_score = }')

local_numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man'
tolerance = 1, ratio_factor = '1.25', stability_factor = '0', snap = 0, suffix = '-opt.npy'

version: bwv253, Tol: 1, RF: 1.25, Average score: 48.4, max score: 82.0 max chord: 58

version: bwv254, Tol: 1, RF: 1.25, Average score: 54.2, max score: 140.0 max chord: 158

version: bwv255, Tol: 1, RF: 1.25, Average score: 50.7, max score: 82.0 max chord: 42

version: bwv256, Tol: 1, RF: 1.25, Average score: 53.6, max score: 140.0 max chord: 138

version: bwv257, Tol: 1, RF: 1.25, Average score: 56.0, max score: 140.0 max chord: 128

version: bwv258, Tol: 1, RF: 1.25, Average score: 57.3, max score: 140.0 max chord: 80

version: bwv259, Tol: 1, RF: 1.25, Average score: 52.7, max score: 104.0 max chord: 74

version: bwv260, Tol: 1, RF: 1.25, Average score: 51.6, max score: 105.0 max chord: 72

version: bwv261, Tol: 1, RF: 1.25, Average score: 53.9, max score: 140.0 max chord: 72

version: bwv262, Tol: 1, RF: 1.25,

In [13]:
# Check that cent tunings have not changed the pitch class of any note
print("Checking pitch class preservation...")
violations_found = False
for tolerance in [1]:
    for ratio_factor in ratio_factors:
        for stability_factor in stability_factors:
            for snap in snaps:
                for suffix in suffixes:
                    for version in ['bwv253', 'bwv254', 'bwv255', 'bwv256', 'bwv257', 'bwv258', 'bwv259', 'bwv260', 'bwv261', 'bwv262', 'bwv263', 'bwv264']:
                        try:
                            # local_numpy_dir = os.path.join(numpy_dir, f't{tolerance}_r{ratio_factor}_s{stability_factor}_md{max_delta}_sn{snap}')
                            # local_numpy_dir = os.path.join(base_dir, 'Archive', 'opt', f'tolerance-{tolerance}' )
                            input_file = os.path.join(local_numpy_dir, f'{version}{suffix}')
                            # input_file = os.path.join(local_numpy_dir, f'{version}-trans-sa-opt.npy') # -trans-sa-opt.npy
                            # print(f'{input_file = }')
                            existing_chorale_in_cents = np.load(input_file)
                        except:
                            print(f'Could not load {input_file}')
                            continue

                        _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)

                        violations = []
                        for chord_inx, (chord_in_cents, chord_12) in enumerate(zip(existing_chorale_in_cents.T, chorale.T)):
                            for voice, (cents, midi) in enumerate(zip(chord_in_cents, chord_12)):
                                original_pc = int(midi) % 12
                                tuned_pc = int(atu.pitch_class_from_cents(cents))  # half-up rounding, consistent with horizontal_transpose.py
                                if original_pc != tuned_pc:
                                    violations.append((chord_inx, voice, int(midi), cents, original_pc, tuned_pc))

                        if violations:
                            violations_found = True
                            print(f'\n{version}: {len(violations)} pitch class violation(s):')
                            for chord_inx, voice, midi, cents, orig_pc, tuned_pc in violations:
                                print(f'  chord {chord_inx}, voice {voice}: MIDI {midi} ({keys[orig_pc]}) -> {cents} cents ({keys[tuned_pc]})')
                        else:
                            print(f'{version}: OK')

if not violations_found:
    print('\nAll pitch classes preserved across all chorales.')

Checking pitch class preservation...

bwv253: 8 pitch class violation(s):
  chord 22, voice 2: MIDI 62 (D♮) -> 259.0 cents (D♯)
  chord 22, voice 3: MIDI 55 (G♮) -> 757.0 cents (G♯)
  chord 23, voice 2: MIDI 62 (D♮) -> 259.0 cents (D♯)
  chord 23, voice 3: MIDI 55 (G♮) -> 757.0 cents (G♯)
  chord 122, voice 1: MIDI 66 (F♯) -> 544.0 cents (F♮)
  chord 123, voice 1: MIDI 66 (F♯) -> 544.0 cents (F♮)
  chord 138, voice 0: MIDI 73 (C♯) -> 35.0 cents (C♮)
  chord 139, voice 0: MIDI 73 (C♯) -> 35.0 cents (C♮)

bwv254: 6 pitch class violation(s):
  chord 142, voice 0: MIDI 67 (G♮) -> 645.0 cents (G♭)
  chord 142, voice 1: MIDI 62 (D♮) -> 147.0 cents (D♭)
  chord 142, voice 3: MIDI 45 (A♮) -> 849.0 cents (A♭)
  chord 143, voice 0: MIDI 67 (G♮) -> 645.0 cents (G♭)
  chord 143, voice 1: MIDI 62 (D♮) -> 147.0 cents (D♭)
  chord 143, voice 3: MIDI 45 (A♮) -> 849.0 cents (A♭)
bwv255: OK

bwv256: 16 pitch class violation(s):
  chord 10, voice 0: MIDI 71 (B♮) -> 965.0 cents (A♯)
  chord 10, voice 1: M

In [7]:
# Analyze the spread of cent values for each pitch class in the original chorale based on the recently tuned numpy arrays of cent values. 
stability_factors = [0]
print(f'{tolerance = }, {ratio_factors = }, {stability_factors = }, {snaps = }, {suffixes = }')
for tolerance in [1]:
    for ratio_factor in ratio_factors:
        for stability_factor in stability_factors:
            for snap in snaps:
                for suffix in suffixes:
                    for horizontal_transpose in [True, False]:
                        # local_numpy_dir = os.path.join(numpy_dir, f't{tolerance}_r{ratio_factor:.2f}_s{stability_factor:.2f}_md{max_delta}_sn{snap}')
                        # local_numpy_dir = os.path.join(base_dir, 'Archive', 'opt', f'tolerance-{tolerance}' )
                        local_numpy_dir = os.path.join(numpy_dir, f't{tolerance}_r{ratio_factor}_s{stability_factor}_md{max_delta}_sn{snap}')
                        print(f'{local_numpy_dir = }, {ratio_factor = }, {stability_factor = }')
                        chorale_in_cents, top_notes, chorale, root, mode, keys = atu.load_chorale_in_cents(version, local_numpy_dir)
                        print(f'Original chorale key: {keys[root]} {mode}')
                        pitch_class_chords = chorale % 12
                        # list all the pitch classes sorted by frequency in the original chorale
                        pitch_class_counts = Counter(pitch_class_chords.flatten())
                        print('Pitch classes sorted by frequency in original chorale:')
                        for note, freq in pitch_class_counts.most_common()[:5]:  # print only the top 5 most common pitch classes
                            print(f'{keys[note]}: {freq}',end='\t')  
                        print()
                        
                        if horizontal_transpose:
                            input_file = os.path.join(local_numpy_dir, f'{version}{suffix}')
                        else: 
                            input_file = os.path.join(local_numpy_dir, f'{version}{suffix}') 
                        cent_value_chorale = np.load(input_file)
                        # print(f'{input_file = }')

                        # Collect every cent value used for each pitch class across the whole chorale
                        pc_cent_values = defaultdict(list)
                        for chord_cents in cent_value_chorale.T:
                            pcs = atu.pitch_class_from_cents(chord_cents)
                            for pc, cv in zip(pcs, chord_cents):
                                pc_cent_values[int(pc)].append(float(cv))

                        # Report spread for the top pitch classes by occurrence frequency
                        print(f'Cent value spread for top pitch classes in {version} with {horizontal_transpose = }')
                        print(f'{"Note":<5} {"Occurs":>7}  {"Min":>6}  {"Max":>6}  {"Range":>6}  Unique cent values (rounded)')
                        print('-' * 90)
                        for note, occurrences in pitch_class_counts.most_common():
                            cvs_raw = pc_cent_values.get(note, [])
                            if not cvs_raw:
                                continue
                            cvs = sorted(set(round(v) for v in cvs_raw))
                            lo, hi = min(cvs), max(cvs)
                            spread = hi - lo
                            # Flag large spreads
                            flag = ' <-- wide spread' if spread > 50 else ''
                            print(f'{keys[note]:<5} {occurrences:>7}  {lo:>6}  {hi:>6}  {spread:>6}  {cvs}{flag}')


tolerance = 1, ratio_factors = array(['1.25'], dtype='<U4'), stability_factors = [0], snaps = array([0]), suffixes = array(['-opt.npy'], dtype='<U8')
local_numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/t1_r1.25_s0_md33_sn0', ratio_factor = '1.25', stability_factor = 0
Original chorale key: A♮ major
Pitch classes sorted by frequency in original chorale:
A♮: 190	E♮: 136	C♯: 100	B♮: 56	F♯: 50	


FileNotFoundError: [Errno 2] No such file or directory: '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/t1_r1.25_s0_md33_sn0/bwv253-opt.npy'